# [실습] LCEL을 이용한 다양한 체인


LangChain Expression Language(LCEL)는 랭체인에서 체인을 간결하게 구성하는 문법입니다.    

단일 체인으로 다양한 모듈을 구성하며, 이 때 `|` 연산자를 사용합니다.

In [ ]:
!pip install langchain==0.3.27 langchain_community==0.3.27 langgraph==0.6.8  langchain-openai langchain-community dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-5-mini',
    temperature = 1.0, 
    max_tokens = 8192
)

llm.invoke("안녕? 너는 모델 이름이 뭐니?")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

print("필수 모듈 임포트 완료")

## LCEL 체인: Prompt | LLM

프롬프트와 LLM을 |로 연결하면, 입력 변수 전달 --> 프롬프트 --> LLM 의 과정이 한 번에 실행됩니다.

매개변수가 2개인 체인도 동일합니다.

## LLM의 구조화된 출력 생성하기

LLM은 기본적으로 텍스트만을 생성하지만, 구조화된 데이터를 생성할 수도 있습니다.   

랭체인의 기본 기능인 with_structured_output을 사용하거나, 파서를 통해 변환합니다.

파서(Parser)는 LLM 뒤에서 출력을 변환합니다.   
이 때, LLM이 파싱할 수 있는 출력을 해야 하므로 프롬프트도 추가합니다.

<br><br>
## Runnables

Runnables는 LCEL의 기본 단위로, 입력을 받아 출력을 생성하는 기본 단위입니다.    
llm, prompt, chain 등이 모두 Runnable 구조에 해당합니다.

이번에는, 데이터 흐름을 제어하는 특별한 Runnable인   
RunnableParallel과 RunnablePassthrough을 이용해 체인을 구성해 보겠습니다.



### RunnableParallel

RunnableParallel은 서로 다른 체인을 병렬적으로 실행합니다.

In [ ]:
from langchain_core.runnables import RunnableParallel

체인의 직렬 연결은 아래와 같이 만들 수 있습니다.

체인의 중간에 Dict가 붙는 경우, 이는 내부적으로 RunnableParallel로 변환되어 실행됩니다.

## RunnableParallel.Assign   

Assign을 사용하면, 직전 체인의 실행 결과를 다음 체인에 전달하고, 결과를 결합할 수 있습니다.   
assign을 붙이기 위해서는 체인의 결과물이 dict 형태여야 합니다.



<br><br>
### RunnablePassthrough
RunnablePassthrough는 체인의 직전 출력을 그대로 가져옵니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough


## 복잡한 체인 만들기
chain2에서 새로운 매개변수가 추가되는 경우는 어떻게 해야 할까요?

<br><br><br>하나의 체인에서 여러 개의 값을 생성하려면,   
JsonOutputParser를 쓰면 됩니다.